# Conventional deep learning: U-Net and SegFormer
# **Protocol (v4).** Train on PEI only (ACI map as the training label; internal PEI train / validation / held-out-east split for model selection). The **final test is the AAFC 2020 field-survey points only**, in PEI, New Brunswick and Ontario, with zero retraining. PEI points are scored twice: all points, and only those in the held-out east test area. ACI-window scores stay as a secondary diagnostic. Same tiles, split, metrics and seeds as the other two notebooks.

| Model | Input |
|---|---|
| U-Net (ResNet34 encoder, ImageNet-initialised) | 6 Sentinel-2 bands x 4 time steps = 24 channels |
| SegFormer-B0 (ImageNet-initialised MiT encoder) | the same 24 channels |

**Before running**
1. Settings: Accelerator = GPU (T4 / P100), Internet = On.
2. Attach the `crop_fm_testAOI_v2` tiles (`*_fm_stack_2020*.tif`) and `PEI_ground_truth_2020_features_FULL*.csv`, `NB_ground_truth_2020_points.csv` and `ON_ground_truth_2020_points.csv` (all three are required; the run stops if one is missing).
3. After the install cell finishes, **restart the session** and continue from the next cell (do not run the install cell again).
4. Save & Run All (Commit) so the run survives disconnects.

In [ ]:
%pip install -q segmentation-models-pytorch rasterio
# IMPORTANT: pip may replace numpy/scipy while the kernel already has them loaded. That gives errors such as
# "numpy._core._multiarray_umath has no attribute _blas_supports_fpe" at `from sklearn...`.
# Fix: after this cell finishes, restart the session (Kaggle: Run > Restart session; Colab: Runtime > Restart session),
# then run from the NEXT cell onward. Do NOT run this install cell again after the restart.
print('Install done. RESTART THE SESSION now, then continue from the next cell.')

In [ ]:
import os, glob, gc, math, random, time, json, re
import numpy as np, pandas as pd, rasterio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import segmentation_models_pytorch as smp

# ---------------- constants ----------------
PATCH, T_STEPS, N_BANDS = 224, 4, 6
N_FEAT = T_STEPS * N_BANDS                # 24 feature bands in the fm tiles; band 25 is the label
N_EMB = 64                                # AlphaEarth embedding bands; band 65 is the label
NODATA, IGNORE, N_CLASSES = -9999.0, 255, 5
AE_NODATA, AE_SCALE = -128, 127.0         # AlphaEarth tiles are int8: value = round(embedding * 127), -128 = no data
CLASS_NAMES = ['Potatoes', 'Pasture and Forages', 'Barley', 'Corn for Grain', 'Wheat']
CODE2IDX = {177: 0, 122: 1, 133: 2, 147: 3, 140: 4}
OUT = '/kaggle/working'

# ---------------- what to run ----------------
RUN_UNET_CONTROL   = True
RUN_SEGFORMER      = True
RUN_PRITHVI_FROZEN = RUN_PRITHVI_FULL = RUN_TERRAMIND = RUN_UNET_1STEP = False
RUN_S2_CLASSICAL   = False
RUN_ALPHAEARTH = RUN_AE_CLASSICAL = RUN_AE_UNET = False     # not part of this notebook
RUN_GROUND_TRUTH   = True
RUN_SCARCITY       = False
SCARCITY_FRACS     = [0.10, 0.25, 0.50]
SEEDS = [42, 43, 44]          # paper numbers
USE_SAVED_WEIGHTS = False

# ---------------- training settings ----------------
EPOCHS, BATCH, ACCUM = 25, 4, 1          # if the GPU runs out of memory: BATCH = 2, ACCUM = 2
LR_ENC, LR_DEC, LR_UNET, LR_SEG = 1e-5, 2e-4, 3e-4, 1e-4
TRAIN_STRIDE = 112
USE_AMP = True
AE_PIX_PER_CLASS = 30000                  # balanced training sample for AlphaEarth + RF/XGBoost

# ---------------- optional Drive download (service account) ----------------
USE_DRIVE = False
DRIVE_FOLDERS = {
    'fm': 'https://drive.google.com/drive/folders/PASTE_crop_fm_testAOI_v2_FOLDER_ID_HERE',
    'ae': 'https://drive.google.com/drive/folders/PASTE_crop_ae_testAOI_v2_FOLDER_ID_HERE',
}
DATA_DIR = '/kaggle/working/data'

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEEDS[0])

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert DEVICE == 'cuda', 'NO GPU FOUND. Settings -> Accelerator -> GPU, then re-run.'

if USE_DRIVE:
    import requests
    from kaggle_secrets import UserSecretsClient
    from google.oauth2 import service_account
    import google.auth.transport.requests
    sa_info = json.loads(UserSecretsClient().get_secret('GDRIVE_SA_KEY'))
    creds = service_account.Credentials.from_service_account_info(sa_info, scopes=['https://www.googleapis.com/auth/drive.readonly'])
    def token():
        if not creds.valid: creds.refresh(google.auth.transport.requests.Request())
        return creds.token
    def download_folder(url):
        folder_id = re.search(r'/folders/([\w-]+)', url).group(1)
        listing, page = [], None
        while True:
            params = {'q': f"'{folder_id}' in parents and trashed=false", 'fields': 'nextPageToken, files(id,name)', 'pageSize': 1000}
            if page: params['pageToken'] = page
            r = requests.get('https://www.googleapis.com/drive/v3/files', params=params, headers={'Authorization': f'Bearer {token()}'}).json()
            if 'error' in r: raise RuntimeError(r['error'].get('message'))
            listing += [f for f in r.get('files', []) if f['name'].lower().endswith(('.tif', '.tiff'))]
            page = r.get('nextPageToken')
            if not page: break
        print(len(listing), 'tif files in', url)
        os.makedirs(DATA_DIR, exist_ok=True)
        for f in listing:
            dest = f'{DATA_DIR}/{f["name"]}'
            if os.path.exists(dest) and os.path.getsize(dest) > 1000: continue
            with requests.get(f'https://www.googleapis.com/drive/v3/files/{f["id"]}', params={'alt': 'media'},
                              headers={'Authorization': f'Bearer {token()}'}, stream=True, timeout=300) as resp:
                resp.raise_for_status()
                with open(dest, 'wb') as fh:
                    for chunk in resp.iter_content(1 << 20): fh.write(chunk)
            print('downloaded', f['name'])
    for kind, url in DRIVE_FOLDERS.items():
        if 'PASTE_' in url:
            print(f'Drive folder for "{kind}" not set - skipped'); continue
        if kind == 'ae' and not RUN_ALPHAEARTH: continue
        download_folder(url)

## 1. Find the tiles and check the band layouts

In [ ]:
def region_of(path):
    base = os.path.basename(path).upper()
    for key in ['PEI', 'NB', 'ON']:
        if base.startswith(key + '_'):
            return key
    return None

def list_tiles(tag):
    """Tiles whose file name contains `tag`, grouped by region; duplicates (same file name in input and Drive dir) kept once."""
    seen, out = set(), {'PEI': [], 'NB': [], 'ON': []}
    for p in sorted(glob.glob('/kaggle/input/**/*.tif', recursive=True) + glob.glob(DATA_DIR + '/**/*.tif', recursive=True)):
        b = os.path.basename(p)
        if tag not in b or b in seen: continue
        r = region_of(p)
        if r: out[r].append(p); seen.add(b)
    return out

files = list_tiles('_fm_stack')
for k, v in files.items():
    print('fm', k, len(v), 'tiles')
assert all(len(v) > 0 for v in files.values()), 'A region has no *_fm_stack* tiles - attach the dataset or set USE_DRIVE = True'
counts = {k: sorted({rasterio.open(p).count for p in v}) for k, v in files.items()}
assert all(c == [N_FEAT + 1] for c in counts.values()), f'Every fm tile must have {N_FEAT + 1} bands, found {counts}'
print('fm band layout OK: 4 timesteps x 6 bands (B2,B3,B4,B8A,B11,B12) + label')

ae_files = list_tiles('_ae_emb') if RUN_ALPHAEARTH else {'PEI': [], 'NB': [], 'ON': []}

def ae_twin(fm_path):
    """The AlphaEarth tile covering exactly the same pixels as this fm tile (same AOI, CRS, scale, fileDimensions)."""
    key = os.path.basename(fm_path).replace('_fm_stack_', '_ae_emb_')
    return next((p for p in ae_files[region_of(fm_path)] if os.path.basename(p) == key), None)

if RUN_ALPHAEARTH:
    problems = []
    for r, v in files.items():
        for p in v:
            t = ae_twin(p)
            if t is None: problems.append(f'no AlphaEarth twin for {os.path.basename(p)}'); continue
            with rasterio.open(p) as a, rasterio.open(t) as b:
                if (a.height, a.width) != (b.height, b.width) or b.count != N_EMB + 1:
                    problems.append(f'{os.path.basename(t)}: shape {b.height}x{b.width}x{b.count}, expected {a.height}x{a.width}x{N_EMB + 1}')
                elif not np.allclose(a.transform, b.transform, atol=1e-3):
                    problems.append(f'{os.path.basename(t)}: grid differs from its fm tile')
    if problems:
        print('AlphaEarth tiles missing or not on the fm grid - AlphaEarth is SKIPPED:')
        for q in problems[:10]: print('  ', q)
        RUN_ALPHAEARTH = False
    else:
        print('AlphaEarth tiles OK: every fm tile has a twin on the identical grid,', rasterio.open(ae_twin(files['PEI'][0])).dtypes[0])

## 2. Patches (224x224). Invalid pixels are stored as NaN, never as -9999

In [ ]:
def read_features(src):
    arr = src.read().astype(np.float32)
    X = arr[:N_FEAT].copy()
    X[~np.isfinite(X) | (X == NODATA)] = np.nan
    lab = arr[N_FEAT]
    lab = np.where(np.isfinite(lab) & (lab != NODATA), lab, IGNORE).astype(np.uint8)
    return X, lab

def patches_from_tile(tif_path, stride, min_label_frac=0.05, min_valid_frac=0.5):
    with rasterio.open(tif_path) as src:
        X, Y = read_features(src); x0 = src.transform.c
    H, W = Y.shape
    out = []
    for i in range(0, H - PATCH + 1, stride):
        for j in range(0, W - PATCH + 1, stride):
            y = Y[i:i+PATCH, j:j+PATCH]; x = X[:, i:i+PATCH, j:j+PATCH]
            if (y != IGNORE).mean() < min_label_frac: continue
            if np.isfinite(x).mean() < min_valid_frac: continue
            out.append({'x': x.astype(np.float16), 'y': y, 'xc': x0 + (j + PATCH / 2) * 30,
                        'tile': tif_path, 'i': i, 'j': j})
    del X, Y; gc.collect()
    return out

def build_region(region, stride):
    pl = []
    for p in files[region]:
        pl += patches_from_tile(p, stride)
    print(f'{region}: {len(pl)} patches from {len(files[region])} tiles (stride {stride})')
    return pl

pei = build_region('PEI', TRAIN_STRIDE)
nb  = build_region('NB', PATCH)
on  = build_region('ON', PATCH)
assert len(pei) > 0 and len(nb) > 0 and len(on) > 0
MIN_TEST_PIXELS = 20000          # below this a class's test score is too noisy to report on its own
for name, pl in [('PEI', pei), ('NB', nb), ('ON', on)]:
    cnt = np.zeros(N_CLASSES)
    for p in pl:
        for c in range(N_CLASSES): cnt[c] += (p['y'] == c).sum()
    print(name, {CLASS_NAMES[c]: f'{int(cnt[c]):,} px ({cnt[c] / cnt.sum():.1%})' for c in range(N_CLASSES)})
    low = [CLASS_NAMES[c] for c in range(N_CLASSES) if cnt[c] < MIN_TEST_PIXELS]
    if name != 'PEI' and low:
        print(f'   !! {name}: {low} under {MIN_TEST_PIXELS:,} labelled pixels - pick a better window with find_test_regions.py')

## 3. Split PEI into train / validation / test (buffered), class weights

In [ ]:
def spatial_split3(patches, test_frac=0.30, val_frac=0.12):
    """West->east split with a buffer of one patch width between parts (overlapping patches cannot leak)."""
    p = sorted(patches, key=lambda d: d['xc'])
    xs = np.array([d['xc'] for d in p]); buf = PATCH * 30
    cut_test = xs[int(len(xs) * (1 - test_frac))]
    train_all = [d for d in p if d['xc'] < cut_test - buf]
    test = [d for d in p if d['xc'] >= cut_test + buf]
    xs2 = np.array([d['xc'] for d in train_all]); cut_val = xs2[int(len(xs2) * (1 - val_frac))]
    train = [d for d in train_all if d['xc'] < cut_val - buf]
    val = [d for d in train_all if d['xc'] >= cut_val]
    return train, val, test

pei_train, pei_val, pei_test = spatial_split3(pei)
print('PEI patches  train / val / test:', len(pei_train), len(pei_val), len(pei_test))

cnt = np.zeros(N_CLASSES)
for p in pei_train:
    for c in range(N_CLASSES): cnt[c] += (p['y'] == c).sum()
class_w = torch.clamp(torch.sqrt(torch.tensor(cnt.sum() / (N_CLASSES * np.maximum(cnt, 1)), dtype=torch.float32)), max=2.0)
print('class pixel counts:', dict(zip(CLASS_NAMES, cnt.astype(int))), '| tempered weights:', class_w.numpy().round(2))

## 4. Normalisation (NaN-aware)

In [ ]:
def fit_band_stats(train_patches, n_sample=150):
    idx = np.random.choice(len(train_patches), min(n_sample, len(train_patches)), replace=False)
    s = np.stack([train_patches[i]['x'].astype(np.float32) for i in idx]).reshape(len(idx), T_STEPS, N_BANDS, PATCH, PATCH)
    mean_, std_, lo_, hi_ = [], [], [], []
    for b in range(N_BANDS):
        v = s[:, :, b].ravel(); v = v[np.isfinite(v)]
        lo, hi = np.percentile(v, [1, 99]); v = np.clip(v, lo, hi)
        mean_.append(v.mean()); std_.append(v.std() + 1e-6); lo_.append(lo); hi_.append(hi)
    f = lambda a: np.array(a, dtype=np.float32)
    return {'mean': f(mean_), 'std': f(std_), 'lo': f(lo_), 'hi': f(hi_)}

def normalise(x24, st):
    """(24,H,W) float with NaN -> (4,6,H,W) normalised, NaN -> 0."""
    x = x24.astype(np.float32).reshape(T_STEPS, N_BANDS, x24.shape[-2], x24.shape[-1])
    bad = ~np.isfinite(x)
    x = np.clip(x, st['lo'][None, :, None, None], st['hi'][None, :, None, None])
    x = (x - st['mean'][None, :, None, None]) / st['std'][None, :, None, None]
    x[bad] = 0.0
    return x

def to_layout(z, layout):
    if layout == 'cthw': return np.ascontiguousarray(z.transpose(1, 0, 2, 3))                              # (6,4,H,W) Prithvi
    if layout == 'flat': return np.ascontiguousarray(z.reshape(T_STEPS * N_BANDS, z.shape[-2], z.shape[-1]))  # (24,H,W) U-Net
    if layout == 'mid':  return np.ascontiguousarray(z[2])                                                  # (6,H,W) one mid-season step
    raise ValueError(layout)

def fm_prep(stats, layout):
    return lambda x: to_layout(normalise(x, stats), layout)

FIT = fit_band_stats(pei_train)

In [ ]:
# SANITY CHECK: normalised valid values must have std ~ 1 with PEI-fit stats; pretrained stats give a narrower spread (reported, not fatal)
samp = np.stack([pei_train[i]['x'].astype(np.float32) for i in np.random.choice(len(pei_train), min(60, len(pei_train)), replace=False)])
valid = np.isfinite(samp.reshape(len(samp), T_STEPS, N_BANDS, PATCH, PATCH))
for nm, st in [('PEI-fit', FIT)]:
    z = np.stack([normalise(s, st) for s in samp])
    sd = [float(z[:, :, b][valid[:, :, b]].std()) for b in range(N_BANDS)]
    mu = [float(z[:, :, b][valid[:, :, b]].mean()) for b in range(N_BANDS)]
    print(f'{nm}: invalid inputs {float((~valid).mean()):.1%} | per-band std {np.round(sd, 2)} | mean {np.round(mu, 2)}')
    if nm == 'PEI-fit':
        assert 0.5 < min(sd) and max(sd) < 1.5, 'PEI-fit normalisation looks broken - do not train'
    elif min(sd) < 0.3:
        print(f'   note: {nm} statistics compress some bands (std < 0.3). Expected with pretraining stats; report it as a domain gap.')
del samp, valid, z; gc.collect()
print('Inputs look healthy.')

## 5. AlphaEarth twins: the same patches, cut from the embedding tiles
The label band of each AlphaEarth tile must match the fm label band pixel for pixel. If it does not, the grids are misaligned and AlphaEarth is skipped.

In [ ]:
def to_ae_int8(a):
    """Raw AlphaEarth bands -> int8 (round(e*127), -128 = no data). Accepts the int8 export or a float export."""
    if a.dtype == np.int8: return a
    a = a.astype(np.float32)
    out = np.full(a.shape, AE_NODATA, dtype=np.int8)
    ok = np.isfinite(a) & (np.abs(a) <= 1.01)
    out[ok] = np.clip(np.round(a[ok] * AE_SCALE), -127, 127).astype(np.int8)
    return out

def ae_decode(a):
    """int8 (or float array holding int8 values / NaN padding) -> float embedding in [-1, 1], NaN = no data."""
    x = a.astype(np.float32)
    x[x == AE_NODATA] = np.nan
    return x / AE_SCALE

def fit_channel_stats(arrs):
    """arrs: list of (C,H,W) float with NaN -> per-channel stats dict."""
    s = np.stack(arrs).astype(np.float32); C = s.shape[1]
    mean_, std_, lo_, hi_ = [], [], [], []
    for c in range(C):
        v = s[:, c].ravel(); v = v[np.isfinite(v)]
        lo, hi = np.percentile(v, [1, 99]); v = np.clip(v, lo, hi)
        mean_.append(v.mean()); std_.append(v.std() + 1e-6); lo_.append(lo); hi_.append(hi)
    f = lambda a: np.array(a, dtype=np.float32)
    return {'mean': f(mean_), 'std': f(std_), 'lo': f(lo_), 'hi': f(hi_)}

def normalise_ch(x, st):
    bad = ~np.isfinite(x)
    x = np.clip(x, st['lo'][:, None, None], st['hi'][:, None, None])
    x = (x - st['mean'][:, None, None]) / st['std'][:, None, None]
    x[bad] = 0.0
    return np.ascontiguousarray(x.astype(np.float32))

if RUN_ALPHAEARTH:
    needed = {id(p): p for pl in [pei_train, pei_val, pei_test, nb, on] for p in pl}
    by_tile = {}
    for p in needed.values(): by_tile.setdefault(p['tile'], []).append(p)
    agree, total = 0, 0
    for tif, plist in by_tile.items():
        with rasterio.open(ae_twin(tif)) as src:
            emb = to_ae_int8(src.read(list(range(1, N_EMB + 1))))
            lab = src.read(N_EMB + 1).astype(np.int16)
        for p in plist:
            i, j = p['i'], p['j']
            p['ae'] = emb[:, i:i+PATCH, j:j+PATCH].copy()
            m = p['y'] != IGNORE
            agree += int((lab[i:i+PATCH, j:j+PATCH][m] == p['y'][m]).sum()); total += int(m.sum())
        del emb, lab; gc.collect()
    print(f'AlphaEarth label agreement with the fm tiles: {agree / total:.4%} of {total} labelled pixels')
    if agree / total < 0.99:
        print('!! The AlphaEarth tiles are not on the same pixel grid as the fm tiles - AlphaEarth is SKIPPED. Re-export with the GEE script.')
        RUN_ALPHAEARTH = False
        for p in needed.values(): p.pop('ae', None)
    else:
        sub = random.Random(0).sample(pei_train, min(60, len(pei_train)))
        AE_ST = fit_channel_stats([ae_decode(p['ae'][:, ::4, ::4]) for p in sub])
        nodata_frac = np.mean([(p['ae'] == AE_NODATA).mean() for p in sub])
        print(f'AlphaEarth no-data fraction in PEI training patches: {nodata_frac:.2%}')
        print('embedding std per band (first 8):', AE_ST['std'][:8].round(3), '-> expected around 0.12 for unit-length 64-D vectors')
    del needed, by_tile; gc.collect()

## 6. Metrics: macro F1 three ways + per-class recall

In [ ]:
def summarise(yt, yp, k=20000, min_support=200, seed=0, min_present=5000, boot=False, n_boot=500):
    """f1_present  : macro F1 over classes with >= min_present reference samples (HEADLINE; tiny classes, e.g. 363 wheat pixels in NB, are too noisy to average in)
       f1_balanced : macro F1 on a class-balanced sample (up to k reference pixels per class, classes with >= min_support);
                     removes the prevalence effect and is comparable with the balanced-sample RF/XGBoost protocol
       f1_all      : macro F1 over all 5 classes, absent ones counting as 0 (the old number; do not report)"""
    yt = yt.astype(np.int64); yp = yp.astype(np.int64)
    rng = np.random.default_rng(seed)
    support = np.bincount(yt, minlength=N_CLASSES)[:N_CLASSES]
    f1c = f1_score(yt, yp, labels=range(N_CLASSES), average=None, zero_division=0)
    rec = np.array([((yp == c) & (yt == c)).sum() / support[c] if support[c] > 0 else np.nan for c in range(N_CLASSES)])
    pres = support >= min_present
    out = {'f1_all': float(f1c.mean()), 'f1_present': float(f1c[pres].mean()), 'support': support,
           'f1_cls': f1c, 'recall': rec, 'f1_balanced': np.nan, 'bal_classes': [], 'n': len(yt),
           'acc': float((yt == yp).mean()),                       # overall accuracy over every class (dominated by the common class)
           'bal_acc': float(np.nanmean(rec[pres])),               # balanced accuracy = mean recall over the classes counted in f1_present
           'f1_classes': [CLASS_NAMES[c] for c in range(N_CLASSES) if pres[c]], 'ci': (np.nan, np.nan)}
    if boot:                                                      # 95% bootstrap interval over the reference samples (field points)
        br = np.random.default_rng(seed + 1); keep_c = np.where(pres)[0]; vals = []
        for _ in range(n_boot):
            ii = br.integers(0, len(yt), len(yt))
            vals.append(f1_score(yt[ii], yp[ii], labels=keep_c, average='macro', zero_division=0))
        out['ci'] = (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))
    keep = [c for c in range(N_CLASSES) if support[c] >= min_support]
    if len(keep) >= 2:
        idx = np.concatenate([rng.choice(np.where(yt == c)[0], min(k, support[c]), replace=False) for c in keep])
        out['f1_balanced'] = float(f1_score(yt[idx], yp[idx], labels=keep, average='macro', zero_division=0))
        out['bal_classes'] = [CLASS_NAMES[c] for c in keep]
    return out

def report(m, yt, yp, tag, unit):
    print(f'\n{tag}   (n {unit} = {m["n"]})')
    print(classification_report(yt, yp, labels=range(N_CLASSES), target_names=CLASS_NAMES, zero_division=0))
    print(f'  Macro F1: present classes = {m["f1_present"]:.4f} (over {", ".join(m["f1_classes"]) or "n/a"}) | balanced sample = {m["f1_balanced"]:.4f} '
          f'(over {", ".join(m["bal_classes"]) or "n/a"}) | all-5 (old) = {m["f1_all"]:.4f}')
    print(f'  Overall accuracy = {m["acc"]:.4f} | balanced accuracy = {m["bal_acc"]:.4f}' +
          (f' | 95% CI of Macro F1 (present) = [{m["ci"][0]:.3f}, {m["ci"][1]:.3f}]' if np.isfinite(m['ci'][0]) else ''))
    print('  recall:', {CLASS_NAMES[c]: (None if np.isnan(m['recall'][c]) else round(float(m['recall'][c]), 2)) for c in range(N_CLASSES)})

rows = []

def add_row(model, cond, m, seed):
    r = {'Model': model, 'Condition': cond, 'Seed': seed, 'N': m['n'],
         'Macro F1 (present)': m['f1_present'], 'Macro F1 (balanced)': m['f1_balanced'], 'Macro F1 (all 5, old)': m['f1_all'],
         'Balanced over': ', '.join(m['bal_classes']), 'F1 counted over': ', '.join(m['f1_classes']),
         'Overall accuracy': m['acc'], 'Balanced accuracy': m['bal_acc'], 'F1 CI low': m['ci'][0], 'F1 CI high': m['ci'][1],
         'Eval set': 'field points' if 'ground truth' in cond else 'ACI windows'}
    for c in range(N_CLASSES):
        r[f'F1 {CLASS_NAMES[c]}'] = m['f1_cls'][c]
        r[f'Recall {CLASS_NAMES[c]}'] = m['recall'][c]
        r[f'Support {CLASS_NAMES[c]}'] = int(m['support'][c])
    rows.append(r)
    pd.DataFrame(rows).to_csv(f'{OUT}/results_fm.csv', index=False)

## 7. Models, loss, training, prediction

In [ ]:
class UNetControl(nn.Module):
    """ImageNet-initialised ResNet34 U-Net with in_ch input channels (24 = 6x4 steps, 6 = one step, 64 = AlphaEarth)."""
    def __init__(self, in_ch):
        super().__init__()
        self.net = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=in_ch, classes=N_CLASSES)
    def forward(self, x): return self.net(x)
    def param_groups(self): return [{'params': list(self.net.parameters()), 'lr': LR_UNET}]

class SegFormerSeg(nn.Module):
    """SegFormer-B0 (ImageNet-pretrained MiT encoder) with in_ch input channels; first patch-embedding layer is re-initialised for in_ch."""
    def __init__(self, in_ch):
        super().__init__()
        from transformers import SegformerForSemanticSegmentation
        self.m = SegformerForSemanticSegmentation.from_pretrained('nvidia/mit-b0', num_labels=N_CLASSES,
                                                                  num_channels=in_ch, ignore_mismatched_sizes=True)
    def forward(self, x):
        lg = self.m(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode='bilinear', align_corners=False)
    def param_groups(self): return [{'params': list(self.parameters()), 'lr': LR_SEG}]

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, target):
        probs = F.softmax(logits, dim=1); valid = (target != IGNORE)
        t = target.clone(); t[~valid] = 0
        onehot = F.one_hot(t, N_CLASSES).permute(0, 3, 1, 2).float(); v = valid.unsqueeze(1).float()
        probs, onehot = probs * v, onehot * v
        inter = (probs * onehot).sum((0, 2, 3)); union = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class ChipDS(Dataset):
    """key = 'x' (fm reflectance) or 'ae' (AlphaEarth int8); prep turns the stored array into the model input."""
    def __init__(self, patches, prep, key='x', augment=False):
        self.p, self.prep, self.key, self.aug = patches, prep, key, augment
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        z = self.prep(self.p[i][self.key]); y = self.p[i]['y'].astype(np.int64)
        if self.aug:
            if np.random.rand() < .5: z, y = z[..., ::-1], y[:, ::-1]
            if np.random.rand() < .5: z, y = z[..., ::-1, :], y[::-1, :]
        return torch.from_numpy(np.ascontiguousarray(z)), torch.from_numpy(np.ascontiguousarray(y))

def _winit(wid): np.random.seed((torch.initial_seed() + wid) % 2**32)

@torch.no_grad()
def predict_patches(model, patches, prep, key):
    model.eval(); yt, yp = [], []
    for x, y in DataLoader(ChipDS(patches, prep, key), batch_size=max(1, BATCH * 2), num_workers=2):
        with torch.autocast(device_type='cuda', enabled=USE_AMP):
            pred = model(x.to(DEVICE)).argmax(1).cpu().numpy()
        y = y.numpy(); m = y != IGNORE
        yt.append(y[m]); yp.append(pred[m])
    return np.concatenate(yt), np.concatenate(yp)

def f1_present(yt, yp):
    labs = np.unique(yt)
    return f1_score(yt, yp, labels=labs, average='macro', zero_division=0)

def fit(model, prep, key, train_p, val_p, tag, seed):
    model.to(DEVICE)
    groups = [g for g in model.param_groups() if len(g['params']) > 0]
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    base = [g['lr'] for g in groups]; warm = 2
    ce = nn.CrossEntropyLoss(weight=class_w.to(DEVICE), ignore_index=IGNORE); dice = DiceLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    dl = DataLoader(ChipDS(train_p, prep, key, augment=True), batch_size=BATCH, shuffle=True, drop_last=True,
                    num_workers=2, worker_init_fn=_winit, generator=torch.Generator().manual_seed(seed))
    best_f1, best_state, best_ep = -1.0, None, 0
    for ep in range(EPOCHS):
        scale = (ep + 1) / warm if ep < warm else 0.5 * (1 + math.cos(math.pi * (ep - warm) / max(1, EPOCHS - warm)))
        for g, b in zip(opt.param_groups, base): g['lr'] = b * scale
        t0 = time.time(); model.train(); tot = 0.0; opt.zero_grad()
        for k, (x, y) in enumerate(dl):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type='cuda', enabled=USE_AMP):
                logits = model(x)
            logits = logits.float()
            loss = 0.5 * ce(logits, y) + 0.5 * dice(logits, y)
            if not torch.isfinite(loss): raise RuntimeError(f'Non-finite loss at epoch {ep+1}. Try USE_AMP = False.')
            scaler.scale(loss / ACCUM).backward(); tot += loss.item()
            if (k + 1) % ACCUM == 0:
                scaler.step(opt); scaler.update(); opt.zero_grad()
        vf1 = f1_present(*predict_patches(model, val_p, prep, key))
        print(f'  [{tag}] epoch {ep+1}/{EPOCHS}  loss {tot/len(dl):.4f}  PEI-val Macro F1 {vf1:.4f}  ({time.time()-t0:.0f}s)')
        if vf1 > best_f1:
            best_f1, best_ep = vf1, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print(f'  [{tag}] best PEI-validation epoch: {best_ep} (Macro F1 {best_f1:.4f}); NB/ON were never used for selection')
    return model

@torch.no_grad()
def predict_tile(model, prep, X):
    """X: (C,H,W) float (NaN = invalid). Non-overlapping 224 windows, NaN padding at the edges. Returns (H,W) class map."""
    model.eval(); C, H, W = X.shape
    Hp, Wp = math.ceil(H / PATCH) * PATCH, math.ceil(W / PATCH) * PATCH
    Xp = np.full((C, Hp, Wp), np.nan, dtype=np.float32); Xp[:, :H, :W] = X
    pred = np.zeros((Hp, Wp), dtype=np.uint8); wins = [(i, j) for i in range(0, Hp, PATCH) for j in range(0, Wp, PATCH)]
    for s in range(0, len(wins), BATCH * 2):
        chunk = wins[s:s + BATCH * 2]
        xb = torch.from_numpy(np.stack([prep(Xp[:, i:i+PATCH, j:j+PATCH]) for i, j in chunk])).to(DEVICE)
        with torch.autocast(device_type='cuda', enabled=USE_AMP):
            pb = model(xb).argmax(1).cpu().numpy()
        for (i, j), p in zip(chunk, pb): pred[i:i+PATCH, j:j+PATCH] = p
    return pred[:H, :W]

## 8. Ground-truth helpers (read the prediction at each AAFC point)

In [ ]:
def load_gt(region_prefix):
    hits = (glob.glob(f'/kaggle/input/**/{region_prefix}_ground_truth_2020_points*.csv', recursive=True)
            or glob.glob(f'/kaggle/input/**/{region_prefix}_ground_truth_2020_features_FULL*.csv', recursive=True))
    if not hits: return None
    print(f'{region_prefix} ground truth from', hits[0])
    df = pd.read_csv(hits[0])
    if 'longitude' not in df.columns or 'latitude' not in df.columns:
        def centre(g):
            ring = np.array(json.loads(g)['coordinates'][0]); return ring[:, 0].mean(), ring[:, 1].mean()
        c = df['.geo'].apply(centre); df['longitude'] = [a for a, b in c]; df['latitude'] = [b for a, b in c]
    df = df[df['crop_class'].isin(CODE2IDX)].copy()
    df['cls'] = df['crop_class'].map(CODE2IDX)
    return df[['longitude', 'latitude', 'cls']].reset_index(drop=True)

def gt_points_in_tile(tif, gt_df):
    from rasterio.warp import transform
    with rasterio.open(tif) as src:
        xs, ys = transform('EPSG:4326', src.crs, list(gt_df['longitude']), list(gt_df['latitude']))
        r, c = rasterio.transform.rowcol(src.transform, xs, ys)
        r, c = np.asarray(r), np.asarray(c)
        inside = (r >= 0) & (r < src.height) & (c >= 0) & (c < src.width)
    return r, c, inside, np.asarray(xs)

GT_MIN_POINTS = 20      # a class enters the point-based Macro F1 only if it has at least this many reference points in that region
                        # (NB: pasture + corn; ON: potatoes, pasture, corn, wheat - barley has 5; PEI: all five)

def gt_collect(region, gt_df, predict_at):
    """predict_at(fm_tile_path, rows, cols) -> predicted classes at those pixels.
    Returns reference classes, predictions and the map x-coordinate (tile CRS, metres) of every point inside the tiles."""
    ys, ps, xx = [], [], []
    seen = np.zeros(len(gt_df), bool)          # each point is scored once, even if tiles/windows touch or overlap
    for tif in files[region]:
        r, c, inside, xs = gt_points_in_tile(tif, gt_df)
        inside = inside & ~seen
        if not inside.any(): continue
        seen |= inside
        ys.append(gt_df['cls'].values[inside]); ps.append(predict_at(tif, r[inside], c[inside])); xx.append(xs[inside])
    if not ys: return None
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(xx)

def gt_score(yt, yp, tag, n_total):
    m = summarise(yt, yp, k=2000, min_support=GT_MIN_POINTS, min_present=GT_MIN_POINTS, boot=True)
    report(m, yt, yp, tag + f'  [{len(yt)} of {n_total} points inside the tiles]', 'ground-truth points')
    if len(yt) < 300: print(f'  !! only {len(yt)} points: treat per-class numbers as noise')
    return m

def gt_validate(region, gt_df, predict_at, tag):
    """Scores the FIELD POINTS of one region. PEI is scored twice: all points (the model saw those pixels' ACI labels in training for part of them)
    and only the points in the held-out east test area (never seen: the independent PEI number)."""
    got = gt_collect(region, gt_df, predict_at)
    if got is None:
        print(f'{tag}: no ground-truth points fall inside the {region} tiles'); return {}
    yt, yp, xs = got
    out = {GT_NAMES[region]: gt_score(yt, yp, tag, len(gt_df))}
    if region == 'PEI':
        t = xs >= PEI_TEST_XMIN
        if t.sum() > 0:
            out[GT_NAMES['PEI_test']] = gt_score(yt[t], yp[t], tag.replace(GT_NAMES['PEI'], GT_NAMES['PEI_test']), len(gt_df))
    return out

def gt_record(name, region, gt_df, predict_at, seed):
    for cond, m in gt_validate(region, gt_df, predict_at, f'{name} | {GT_NAMES[region]}').items():
        add_row(name, cond, m, seed)

def fm_reader(tif):
    with rasterio.open(tif) as src: return read_features(src)[0]

def ae_reader(tif):
    with rasterio.open(ae_twin(tif)) as src: return to_ae_int8(src.read(list(range(1, N_EMB + 1)))).astype(np.float32)

def dl_predict_at(model, prep, reader):
    def f(tif, r, c):
        return predict_tile(model, prep, reader(tif))[r, c]
    return f

GT = {r: load_gt(r) for r in ['PEI', 'NB', 'ON']} if RUN_GROUND_TRUTH else {'PEI': None, 'NB': None, 'ON': None}
GT_NAMES = {'PEI': 'PEI vs AAFC ground truth (all points, in-region)', 'PEI_test': 'PEI vs AAFC ground truth (held-out east test area)',
            'NB': 'PEI -> NB vs AAFC ground truth', 'ON': 'PEI -> Ontario vs AAFC ground truth'}
print({r: (None if d is None else len(d)) for r, d in GT.items()}, '<- ground-truth points per region (None = file not attached)')
if RUN_GROUND_TRUTH:
    missing = [r for r, d in GT.items() if d is None]
    assert not missing, (f'Ground-truth file missing for {missing}. Attach PEI_ground_truth_2020_features_FULL*.csv (or *_points.csv), '
                         'NB_ground_truth_2020_points.csv and ON_ground_truth_2020_points.csv. The field points are the FINAL TEST, so the run must not continue without them.')
# x of the first held-out PEI test patch's left edge (tile CRS, metres): field points east of this were never inside any training/validation patch
PEI_TEST_XMIN = min(d['xc'] for d in pei_test) - PATCH * 30 / 2
print('PEI held-out test area starts at x =', round(PEI_TEST_XMIN), 'm (EPSG:3979)')
for r_, d_ in GT.items():
    print(r_, 'reference points per class:', {CLASS_NAMES[c]: int((d_['cls'] == c).sum()) for c in range(N_CLASSES)} if d_ is not None else None)

## 9. Model runners

In [ ]:
RUN_ACI_WINDOWS = True      # True = also score the ACI-map windows (secondary / diagnostic). The FINAL TEST is always the field points.
CONDITIONS = [('PEI in-domain test', lambda: pei_test), ('PEI -> New Brunswick', lambda: nb), ('PEI -> Ontario', lambda: on)] if RUN_ACI_WINDOWS else []

def safe(name): return re.sub(r'[^A-Za-z0-9._-]+', '_', name).strip('_')

def find_saved(name, seed):
    for fname in [f'{safe(name)}__seed{seed}.pt'] + ([name.replace(' ', '_').replace('/', '-') + '.pt'] if seed == 42 else []):
        hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if hits: return hits[0]
    return None

def run_dl(spec, train_p, seed, prefix='', save=True, do_gt=True):
    name = spec['name']
    print(f'\n{"=" * 78}\n{prefix}{name} | seed {seed} | {len(train_p)} training patches\n{"=" * 78}')
    set_seed(seed)
    model = spec['build']()
    saved = find_saved(name, seed) if (USE_SAVED_WEIGHTS and not prefix) else None
    if saved:
        model.load_state_dict(torch.load(saved, map_location='cpu')); model.to(DEVICE)
        print('  loaded saved weights', saved, '(no training)')
    else:
        model = fit(model, spec['prep'], spec['key'], train_p, pei_val, f'{prefix}{name}', seed)
        if save and not prefix: torch.save(model.state_dict(), f'{OUT}/{safe(name)}__seed{seed}.pt')
    for cond, data in CONDITIONS:
        yt, yp = predict_patches(model, data(), spec['prep'], spec['key'])
        m = summarise(yt, yp); report(m, yt, yp, f'{prefix}{name} | {cond}', 'pixels')
        add_row(name, prefix + cond, m, seed)
    if do_gt and RUN_GROUND_TRUTH and not prefix:
        for r in ['PEI', 'NB', 'ON']:
            if GT[r] is None: continue
            gt_record(name, r, GT[r], dl_predict_at(model, spec['prep'], spec['reader']), seed)
    del model; gc.collect(); torch.cuda.empty_cache()

# ---- AlphaEarth + RandomForest / XGBoost (pixel-wise, same pixels as every other model) ----
def ae_pixels(p):
    m = p['y'] != IGNORE
    X = ae_decode(p['ae'])[:, m].T
    return np.nan_to_num(X, nan=0.0), p['y'][m]

def ae_train_sample(patches, k, seed, pix=None):
    pix = pix or ae_pixels
    rng = np.random.default_rng(seed); per_patch = max(50, 4 * k // max(1, len(patches)))
    Xs, ys = [], []
    for p in patches:
        X, y = pix(p)
        for c in range(N_CLASSES):
            idx = np.where(y == c)[0]
            if len(idx): idx = rng.choice(idx, min(per_patch, len(idx)), replace=False); Xs.append(X[idx]); ys.append(y[idx])
    X, y = np.concatenate(Xs), np.concatenate(ys)
    keep = np.concatenate([rng.choice(np.where(y == c)[0], min(k, (y == c).sum()), replace=False) for c in range(N_CLASSES) if (y == c).any()])
    return X[keep], y[keep]

def ae_make(kind, seed):
    if kind == 'RandomForest':
        return RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=seed)
    return XGBClassifier(n_estimators=300, tree_method='hist', device='cuda', eval_metric='mlogloss', random_state=seed)

def run_ae_classical(seed, train_p=None, prefix=''):
    train_p = train_p or pei_train
    X, y = ae_train_sample(train_p, AE_PIX_PER_CLASS, seed)
    print(f'\n{"=" * 78}\n{prefix}AlphaEarth + RF/XGBoost | seed {seed} | balanced sample {np.bincount(y, minlength=N_CLASSES)}\n{"=" * 78}')
    for kind in ['RandomForest', 'XGBoost']:
        name = f'AlphaEarth 2020 + {kind}'
        t0 = time.time(); clf = ae_make(kind, seed).fit(X, y); print(f'  {name} trained in {time.time()-t0:.0f}s')
        for cond, data in CONDITIONS:
            yt, yp = [], []
            for p in data():
                Xp, ypp = ae_pixels(p); yt.append(ypp); yp.append(clf.predict(Xp))
            yt, yp = np.concatenate(yt), np.concatenate(yp)
            m = summarise(yt, yp); report(m, yt, yp, f'{prefix}{name} | {cond}', 'pixels')
            add_row(name, prefix + cond, m, seed)
        if RUN_GROUND_TRUTH and not prefix:
            def at(tif, r, c, clf=clf):
                with rasterio.open(ae_twin(tif)) as src:
                    a = to_ae_int8(src.read(list(range(1, N_EMB + 1))))
                return clf.predict(np.nan_to_num(ae_decode(a[:, r, c]).T, nan=0.0))
            for rg in ['PEI', 'NB', 'ON']:
                if GT[rg] is None: continue
                gt_record(name, rg, GT[rg], at, seed)
        del clf; gc.collect()

# ---- Sentinel-2 + RandomForest / XGBoost (pixel-wise, same pixels, same split, same metrics as every other model) ----
def s2_features(x24):
    """(24,H,W) reflectance (NaN = invalid) -> (32,H,W): the 24 bands + NDVI and NDWI for each of the 4 steps."""
    x = x24.astype(np.float32).reshape(T_STEPS, N_BANDS, x24.shape[-2], x24.shape[-1])
    green, red, nir = x[:, 1], x[:, 2], x[:, 3]
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = (nir - red) / (nir + red); ndwi = (green - nir) / (green + nir)
    return np.concatenate([x24.astype(np.float32), ndvi, ndwi], axis=0)

S2_FILL = None
def s2_pixels(p):
    m = p['y'] != IGNORE
    X = s2_features(p['x'])[:, m].T
    return np.where(np.isfinite(X), X, S2_FILL), p['y'][m]

def run_s2_classical(seed, train_p=None, prefix=''):
    global S2_FILL
    train_p = train_p or pei_train
    f = np.concatenate([s2_features(p['x'])[:, ::8, ::8].reshape(T_STEPS * N_BANDS + 2 * T_STEPS, -1) for p in train_p[:60]], axis=1)
    S2_FILL = np.nanmedian(np.where(np.isfinite(f), f, np.nan), axis=1).astype(np.float32)
    X, y = ae_train_sample(train_p, AE_PIX_PER_CLASS, seed, pix=s2_pixels)
    print(f'\n{"=" * 78}\n{prefix}Sentinel-2 + RF/XGBoost | seed {seed} | balanced sample {np.bincount(y, minlength=N_CLASSES)}\n{"=" * 78}')
    for kind in ['RandomForest', 'XGBoost']:
        name = f'Sentinel-2 (24 bands + NDVI/NDWI) + {kind}'
        t0 = time.time(); clf = ae_make(kind, seed).fit(X, y); print(f'  {name} trained in {time.time()-t0:.0f}s')
        for cond, data in CONDITIONS:
            yt, yp = [], []
            for p in data():
                Xp, ypp = s2_pixels(p); yt.append(ypp); yp.append(clf.predict(Xp))
            yt, yp = np.concatenate(yt), np.concatenate(yp)
            m = summarise(yt, yp); report(m, yt, yp, f'{prefix}{name} | {cond}', 'pixels')
            add_row(name, prefix + cond, m, seed)
        if RUN_GROUND_TRUTH and not prefix:
            def at(tif, r, c, clf=clf):
                F_ = s2_features(fm_reader(tif))[:, r, c].T
                return clf.predict(np.where(np.isfinite(F_), F_, S2_FILL))
            for rg in ['PEI', 'NB', 'ON']:
                if GT[rg] is None: continue
                gt_record(name, rg, GT[rg], at, seed)
        del clf; gc.collect()

# ---- the deep-learning specs ----
DL_SPECS = []
if RUN_UNET_CONTROL:   DL_SPECS.append(dict(name='U-Net (ResNet34, 6 bands x 4 steps)', build=lambda: UNetControl(N_FEAT),
                                             prep=fm_prep(FIT, 'flat'), key='x', reader=fm_reader))
if RUN_SEGFORMER:      DL_SPECS.append(dict(name='SegFormer-B0 (6 bands x 4 steps)', build=lambda: SegFormerSeg(N_FEAT),
                                             prep=fm_prep(FIT, 'flat'), key='x', reader=fm_reader))
if RUN_ALPHAEARTH and RUN_AE_UNET:
    DL_SPECS.append(dict(name='AlphaEarth 2020 + UNet head', build=lambda: UNetControl(N_EMB),
                         prep=lambda a: normalise_ch(ae_decode(a), AE_ST), key='ae', reader=ae_reader))
print('Will run:', [s['name'] for s in DL_SPECS] + (['AlphaEarth 2020 + RandomForest', 'AlphaEarth 2020 + XGBoost']
      if RUN_ALPHAEARTH and RUN_AE_CLASSICAL else []) + (['Sentinel-2 + RandomForest', 'Sentinel-2 + XGBoost'] if RUN_S2_CLASSICAL else []), '| seeds', SEEDS)

## 10. Run everything (once per seed)
TerraMind is the only run allowed to fail: TerraTorch's TerraMind API has changed between releases. If it fails, the error is printed and the rest continues; send me the message.

In [ ]:
for seed in SEEDS:
    for spec in DL_SPECS:
        try:
            run_dl(spec, pei_train, seed)
        except Exception as e:
            if not spec.get('optional'): raise
            print(f'!! {spec["name"]} failed: {e!r}'); gc.collect(); torch.cuda.empty_cache()
    if RUN_ALPHAEARTH and RUN_AE_CLASSICAL:
        run_ae_classical(seed)
    if RUN_S2_CLASSICAL:
        run_s2_classical(seed)

## 11. Optional: label-scarcity study
Trains on 10 / 25 / 50 % of the PEI training patches (the same subset for every model). If pretraining helps, the gap to the U-Net control should be largest at low label counts.

In [ ]:
if RUN_SCARCITY:
    for frac in SCARCITY_FRACS:
        sub = random.Random(SEEDS[0]).sample(pei_train, max(8, int(frac * len(pei_train))))
        pre = f'{int(frac * 100)}% labels | '
        for spec in DL_SPECS:
            try: run_dl(spec, sub, SEEDS[0], prefix=pre, do_gt=False)
            except Exception as e:
                if not spec.get('optional'): raise
                print(f'!! {spec["name"]} failed: {e!r}')
        if RUN_ALPHAEARTH and RUN_AE_CLASSICAL: run_ae_classical(SEEDS[0], train_p=sub, prefix=pre)
        if RUN_S2_CLASSICAL: run_s2_classical(SEEDS[0], train_p=sub, prefix=pre)

## 12. Results
- `results_fm.csv`: one row per model x condition x seed (all metrics, per-class recall and support).
- `results_fm_summary.csv`: mean and std over seeds. **Report "Macro F1 (present)" as the headline**, "Macro F1 (balanced)" when comparing with the RF/XGBoost tier, and the recall table for transfer.
- Single-seed differences below about 0.03 are not meaningful.

In [ ]:
res = pd.DataFrame(rows); res.to_csv(f'{OUT}/results_fm.csv', index=False)
METRICS = ['Macro F1 (present)', 'Overall accuracy', 'Balanced accuracy', 'Macro F1 (balanced)', 'Macro F1 (all 5, old)']
summ = res.groupby(['Model', 'Condition'])[METRICS].agg(['mean', 'std', 'count'])
summ.columns = [f'{a} {b}' for a, b in summ.columns]
summ.reset_index().to_csv(f'{OUT}/results_fm_summary.csv', index=False)

pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20)
fp = res[res['Eval set'] == 'field points']
print('=' * 30, 'FINAL TEST: AAFC 2020 field points (mean over seeds)', '=' * 30)
for col in ['Macro F1 (present)', 'Overall accuracy', 'Balanced accuracy']:
    print(f'\n{col}'); print(fp.pivot_table(index='Model', columns='Condition', values=col, aggfunc='mean').round(3).to_string())
fp.groupby(['Model', 'Condition'])[['Macro F1 (present)', 'Overall accuracy', 'Balanced accuracy', 'F1 CI low', 'F1 CI high']].agg(['mean', 'std']).to_csv(f'{OUT}/results_fieldpoints_summary.csv')
print('\nSaved results_fieldpoints_summary.csv')
for col in METRICS:
    print(f'\n{col}  (mean over {len(SEEDS)} seed(s))')
    print(res.pivot_table(index='Condition', columns='Model', values=col, aggfunc='mean').round(3).to_string())
    if len(SEEDS) > 1:
        print(f'{col}  std over seeds')
        print(res.pivot_table(index='Condition', columns='Model', values=col, aggfunc='std').round(3).to_string())

rec_cols = [f'Recall {c}' for c in CLASS_NAMES]
rec = res.groupby(['Condition', 'Model'])[rec_cols].mean().round(2)
rec.to_csv(f'{OUT}/results_fm_recall.csv')
print('\nPer-class recall (mean over seeds)'); print(rec.to_string())
print('\nSaved: results_fm.csv, results_fm_summary.csv, results_fm_recall.csv and the .pt weights in /kaggle/working')